# ACTIVIDAD SESIÓN 3: ELEMENTOS BÁSICOS DE SPARK

Una plataforma de *streaming* desea analizar cuáles son las películas más vistas en Chile durante el último mes.  
Para esto, se dispone de un dataset con información sobre las visualizaciones de películas, incluyendo:  
- Usuario  
- Nombre de la película  
- Cantidad de minutos vistos  
- Calificación otorgada (*rating*)  
- Género

**Objetivo**  
Procesar estos datos utilizando **Apache Spark y RDDs**, aplicando transformaciones y acciones para obtener información relevante.

**Dataset**: `peliculas_mas_vistas.csv`  
Cada fila representa una visualización de una película por un usuario.

Ejemplo:

| usuario | película                     | minutos_vistos | rating | género      |
|---------|------------------------------|----------------|--------|-------------|
| Juan    | La Gran Aventura             | 120            | 4.5    | Animación   |
| Ana     | Acción Extrema               | 90             | 3.8    | Acción      |
| Pedro   | La Gran Aventura             | 110            | 4.2    | Animación   |
| Carla   | Drama Profundo               | 150            | 4.9    | Drama       |
| Luis    | Documental de la Naturaleza  | 95             | 4.1    | Documental  |


## 1. Carga y Preprocesamiento de Datos (1 punto)
- Cargar el dataset en un RDD.  
- Eliminar la primera fila (encabezado).  
- Convertir los datos a tuplas: `(usuario, película, minutos_vistos, rating, género)`.


In [1]:
from pyspark.sql import SparkSession

# Crear sesión de Spark
spark = SparkSession.builder.appName("PeliculasChile").getOrCreate()
sc = spark.sparkContext

print("Spark inicializado")
print("Versión:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/05 21:34:08 WARN Utils: Your hostname, Matheus-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.94 instead (on interface en0)
25/09/05 21:34:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 21:34:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/05 21:34:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark inicializado
Versión: 4.0.0


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 52985)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/Library/Frameworks/Pyt

In [2]:
# Crear dataset de ejemplo directamente
data = [
    "usuario,pelicula,minutos_vistos,rating,genero",
    "Juan,La Gran Aventura,120,4.5,Animación",
    "Ana,Acción Extrema,90,3.8,Acción",
    "Pedro,La Gran Aventura,110,4.2,Animación",
    "Carla,Drama Profundo,150,4.9,Drama",
    "Juan,Acción Extrema,85,4.0,Acción",
    "Ana,Romance Inesperado,130,4.7,Romance",
    "Luis,Documental de la Naturaleza,95,4.1,Documental",
    "Pedro,Romance Inesperado,120,4.4,Romance",
    "Carla,Acción Extrema,100,4.2,Acción",
    "Luis,Drama Profundo,140,4.8,Drama"
]

rdd_raw = sc.parallelize(data)
header = rdd_raw.first()
rdd = rdd_raw.filter(lambda x: x != header).map(lambda line: line.split(","))
rdd = rdd.map(lambda x: (x[0], x[1], int(x[2]), float(x[3]), x[4]))

rdd.take(5)

[('Juan', 'La Gran Aventura', 120, 4.5, 'Animación'),
 ('Ana', 'Acción Extrema', 90, 3.8, 'Acción'),
 ('Pedro', 'La Gran Aventura', 110, 4.2, 'Animación'),
 ('Carla', 'Drama Profundo', 150, 4.9, 'Drama'),
 ('Juan', 'Acción Extrema', 85, 4.0, 'Acción')]

25/09/05 23:18:52 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 840379 ms exceeds timeout 120000 ms
25/09/05 23:18:52 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/05 23:18:53 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

## 2. Cantidad de Visualizaciones por Película (1 punto)
- Contar cuántas veces ha sido vista cada película.


In [ ]:
# 2️⃣ Cantidad de visualizaciones por película
views_per_movie = rdd.map(lambda x: (x[1], 1))
views_per_movie=views_per_movie.collect()
print(f"poniendo umero uno a todas las lineas {views_per_movie}")


In [ ]:

views_per_movie=rdd.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a+b)
views_per_movie=views_per_movie.collect()
print(f" sumando las peliculas, para dejarlo de manera unica {views_per_movie}")

## 3. Tiempo Total de Visualización por Película (1 punto)
- Sumar el total de minutos vistos por cada película.  
- Mostrar el **Top 3** de las más vistas.


In [ ]:
# 3) Tiempo total de visualización por película (Top 3)
minutes_per_movie = rdd.map(lambda x: (x[1], x[2])).reduceByKey(lambda a, b: a+b)
top3_movies = minutes_per_movie.takeOrdered(3, key=lambda x: -x[1])
top3_movies

## 4. Películas con un Rating Promedio Mayor a 4.5 (1 punto)
- Calcular el **rating promedio** de cada película.  
- Filtrar las que tengan un promedio **mayor a 4.5**.


In [ ]:
# 4️) Películas con rating promedio mayor a 4.5
ratings = rdd.map(lambda x: (x[1], (x[3], 1)))
ratings_avg = ratings.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
ratings_avg = ratings_avg.mapValues(lambda v: v[0]/v[1])
ratings_avg.filter(lambda x: x[1] > 4.5).collect()

## 5. Promedio de Minutos Vistos por Género (1 punto)
- Obtener el **tiempo promedio de visualización** por cada género.


In [ ]:
# 5️) Promedio de minutos vistos por género
minutes_by_genre = rdd.map(lambda x: (x[4], (x[2], 1)))
minutes_avg = minutes_by_genre.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
minutes_avg.mapValues(lambda v: v[0]/v[1]).collect()

## 6. Usuarios con Mayor Tiempo de Visualización Acumulado (1 punto)
- Calcular el tiempo total visto por usuario.  
- Mostrar los **3 usuarios** con más tiempo de visualización.


In [ ]:
# 6️⃣ Usuarios con mayor tiempo de visualización acumulado (Top 3)
user_time = rdd.map(lambda x: (x[0], x[2])).reduceByKey(lambda a, b: a+b)
user_time.takeOrdered(3, key=lambda x: -x[1])

## 7. Género Más Popular (1 punto)
- Determinar el género con más visualizaciones en total.


In [ ]:
# 7️) Género más popular (más visualizaciones)
genre_views = rdd.map(lambda x: (x[4], 1)).reduceByKey(lambda a, b: a+b)
genre_views.takeOrdered(1, key=lambda x: -x[1])

## 8. Película con Mayor Rating en Cada Género (1 punto)
- Para cada género, obtener la **película con mayor rating promedio**.


In [ ]:
# 8️⃣ Película con mayor rating promedio en cada género
genre_movie_rating = rdd.map(lambda x: ((x[4], x[1]), (x[3], 1)))
genre_movie_avg = genre_movie_rating.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
genre_movie_avg = genre_movie_avg.mapValues(lambda v: v[0]/v[1])

genre_best = genre_movie_avg.map(lambda x: (x[0][0], (x[0][1], x[1])))\
    .reduceByKey(lambda a, b: a if a[1] > b[1] else b)
genre_best.collect()

## 9. Distribución de Ratings (1 punto)
- Contar cuántas películas tienen un rating en los rangos:  
  - 1–2  
  - 2–3  
  - 3–4  
  - 4–5


In [ ]:
# 9️) Distribución de ratings
rating_ranges = rdd.map(lambda x: (x[1], x[3]))

def bucket(r):
    if 1 <= r < 2: return "1-2"
    elif 2 <= r < 3: return "2-3"
    elif 3 <= r < 4: return "3-4"
    elif 4 <= r <= 5: return "4-5"

rating_dist = rating_ranges.map(lambda x: (bucket(x[1]), 1)).reduceByKey(lambda a, b: a+b)
rating_dist.collect()

## 10. Optimización y Explicación del Código (1 punto)
- Explicar brevemente el uso de **lazy evaluation** en Spark.  

Lazy evaluation es para utilizar recursos solamente cuando sea necesario, para evitar el uso innecesario de recursos computacionales

- Indicar cómo optimizar el código usando `persist()` o `cache()`.

La idea de guardar en el disco o en el cache es según la cantidad de usos que le darás a la variable, es decir si vas a utilizarlo muchas veces, deberias guardar las variables en cache, pero sin embargo se guarda de manera temporal.
si deseas utilizar las variables nuevamente pero después de varios días, 

## INSTRUCCIONES ADICIONALES
- **Puntos totales = 10**.  
- Comprimir el archivo en `.zip` o `.rar`.  
- Subir el archivo a la plataforma.
